# Download data and extract Perch embeddings

Run once before training notebooks.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import importlib.util
import os
import sys
from pathlib import Path

def _load_project():
    my_drive = Path("/content/drive/MyDrive")
    init_candidates = [
        p for p in my_drive.rglob("colab_init.py")
        if (p.parent / "birdclef").is_dir()
    ]
    if init_candidates:
        init_path = min(init_candidates, key=lambda p: len(p.parts))
        spec = importlib.util.spec_from_file_location("colab_init", init_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return
    roots = [
        nb.parent.resolve()
        for nb in my_drive.rglob("02_download_and_extract_embeddings.ipynb")
        if (nb.parent / "birdclef").is_dir()
    ]
    if not roots:
        raise FileNotFoundError(
            "Unzip the full repository on Google Drive, then open a notebook from that folder."
        )
    root = min(roots, key=lambda p: len(p.parts))
    sys.path.insert(0, str(root))
    os.chdir(root)

_load_project()

!pip install -q kaggle huggingface_hub onnxruntime-gpu soundfile tqdm

from birdclef.colab import mount_and_configure, require_kaggle_token

mount_and_configure()
require_kaggle_token()


## Download BirdCLEF competition data

In [ ]:
import shutil
import subprocess
from pathlib import Path

import birdclef.paths as paths

RAW = Path("/content/bird_data")
RAW.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["kaggle", "competitions", "download", "-c", "birdclef-2026", "-p", str(RAW)],
    check=True,
)
zip_path = RAW / "birdclef-2026.zip"
if zip_path.exists():
    subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(RAW)], check=True)

paths.METADATA_DIR.mkdir(parents=True, exist_ok=True)
for name in ("train.csv", "sample_submission.csv"):
    src = RAW / name
    if src.exists():
        shutil.copy(src, Path("/content") / name)
        shutil.copy(src, paths.METADATA_DIR / name)
        print(f"Staged {name}")

## Download Perch v2 ONNX (Hugging Face)

Community ONNX export of [Google Perch v2](https://www.kaggle.com/models/google/bird-vocalization-classifier/) from [justinchuby/Perch-onnx](https://huggingface.co/justinchuby/Perch-onnx) (~394 MB). Downloaded once and cached on Drive.


In [ ]:
from birdclef.colab import ensure_perch_onnx

ensure_perch_onnx()


## Baseline embeddings (5 s windows)


In [ ]:
import pandas as pd
from birdclef.extract import extract_baseline_embeddings
import birdclef.paths as paths

save_dir = str(paths.EMBEDDINGS_ARCHIVE_DIR)

df = pd.read_csv(paths.TRAIN_CSV)
extract_baseline_embeddings(df, str(paths.PERCH_ONNX), str(paths.TRAIN_AUDIO_DIR), save_dir)
print(f"Baseline embeddings saved to {save_dir}")


## TTA embeddings (2.5 s stride)


In [ ]:
from birdclef.extract import extract_tta_embeddings
import birdclef.paths as paths

save_dir = str(paths.EMBEDDINGS_TTA_ARCHIVE_DIR)

extract_tta_embeddings(df, str(paths.PERCH_ONNX), str(paths.TRAIN_AUDIO_DIR), save_dir)
print(f"TTA embeddings saved to {save_dir}")


## Optional: archive embeddings to Drive

Skip if archives already exist. Re-run after a fresh extraction to rebuild zips.


In [ ]:
import shutil
from pathlib import Path

import birdclef.paths as paths

for folder, archive_name in (
    (paths.EMBEDDINGS_ARCHIVE_DIR, "embeddings_v2_archive.zip"),
    (paths.EMBEDDINGS_TTA_ARCHIVE_DIR, "embeddings_v2_TTA_archive.zip"),
):
    local_zip = Path("/content") / archive_name.replace(".zip", "")
    shutil.make_archive(str(local_zip), "zip", str(folder))
    shutil.move(str(local_zip) + ".zip", str(paths.PROJECT_ROOT / archive_name))
    print(f"Archived {folder} to {paths.PROJECT_ROOT / archive_name}")